In [419]:
import sys
sys.path.insert(0, '../')

import re
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)

In [420]:
def _str_to_interval(s):
    m = re.match(r'\[([-\d.]+),\s*([-\d.inf]+)\)', str(s))
    if m:
        return pd.Interval(float(m.group(1)), float(m.group(2)), closed='left')
    return s

In [440]:
def _collapse_to_health_bracket(iv):
    """Map a 5-year age bracket to the matching health-survey bracket and the
    fraction of that bracket which falls inside it. Returns (target, weight)
    or None for brackets below 18 that have no health counterpart."""
    if iv == pd.Interval(15.0, 20.0, closed='left'):
        # health starts at 18, so only ages 18-19 of the [15, 20) span count
        return pd.Interval(18.0, 35.0, closed='left'), 2 / 5
    left = iv.left
    if left < 18:
        return None
    if left < 35:
        return pd.Interval(18.0, 35.0, closed='left'), 1.0
    if left < 50:
        return pd.Interval(35.0, 50.0, closed='left'), 1.0
    if left < 65:
        return pd.Interval(50.0, 65.0, closed='left'), 1.0
    return pd.Interval(65.0, np.inf, closed='left'), 1.0

In [441]:
data_age = pd.read_csv('../data/bayesian_network/age-fixed.csv')
data_health = pd.read_csv('../data/bayesian_network/health-fixed.csv')

data_age = data_age.rename(columns = {'gender': 'sex'})
data_age = data_age.set_index(['sex', 'age_group'])
data_age.index = data_age.index.set_levels(
    data_age.index.levels[data_age.index.names.index('age_group')].map(_str_to_interval),
    level='age_group'
)

data_health = data_health.set_index(['sex', 'age_group'])
data_health.index = data_health.index.set_levels(
    data_health.index.levels[data_health.index.names.index('age_group')].map(_str_to_interval),
    level='age_group'
)
data_health = data_health.sort_index()

In [446]:
_mapped = [_collapse_to_health_bracket(iv) for iv in data_age.index.get_level_values('age_group')]
_keep = [m is not None for m in _mapped]

_weights = np.array([m[1] for m in _mapped if m is not None])
_target_age = [m[0] for m in _mapped if m is not None]
_sex = data_age.index.get_level_values('sex')[_keep]

data_age = (
    data_age[_keep]
    .mul(_weights, axis=0)
    .set_axis(pd.MultiIndex.from_arrays([_sex, _target_age], names=['sex', 'age_group']))
    .groupby(level=['sex', 'age_group'])
    .sum()
    .round()
    .astype(int)
    .sort_index()
)

# The Canada total is a derived sum, not an independent measurement: rounding each
# province column on its own can leave it off by 1 (only the [18, 35) bracket is
# fractional, via the 2/5 weight). Recompute it from the rounded provinces so it stays consistent.
data_age['Canada (excluding territories)'] = (
    data_age.drop(columns='Canada (excluding territories)').sum(axis=1)
)

---------------------------

In [447]:
age_provinces = data_age.iloc[:, :-1]
age_canada = data_age['Canada (excluding territories)']

# P(location) - Probability of being from a province
p_loc = (age_provinces / age_canada.sum()).sum()

# P(sex|location)
p_sex_given_loc = age_provinces.groupby('sex').sum() / age_provinces.sum()

# P(age | sex, loc)
p_age_given_sexloc = age_provinces.groupby('sex').apply(lambda g: g / g.sum())

# P(condition | sex, age)
p_cond_given_sexage = data_health.div(age_canada, axis=0)

In [449]:
def sample_single():
    loc = rng.choice(p_loc.index, p=p_loc.values)

    sexes = p_sex_given_loc[loc]
    sex = rng.choice(p_sex_given_loc.index, p=sexes.values)

    ages = p_age_given_sexloc[loc][sex]
    age = rng.choice(ages.index.get_level_values(1), p=ages.values)

    conds = p_cond_given_sexage.loc[(sex, age)]
    obese      = rng.random() < conds['Obese']
    smoker     = rng.random() < conds['Current smoker']
    hbp        = rng.random() < conds['High blood pressure']
    vaccinated = rng.random() < conds['Recently vaccinated']

    return (loc, sex, age, obese, smoker, hbp, vaccinated)

In [450]:
pd.DataFrame(
    [sample_single() for _ in range(10_000)],
    columns = ['location', 'sex', 'age', 'obese', 'smoker', 'hbp', 'vaccinated']
)

,location,sex,age,obese,smoker,hbp,vaccinated
0,Ontario,Female,"[18.0, 35.0)",True,False,False,False
1,Ontario,Male,"[65.0, inf)",False,True,False,True
2,Ontario,Female,"[65.0, inf)",False,False,True,True
3,British Columbia,Male,"[50.0, 65.0)",False,False,False,False
4,Ontario,Male,"[50.0, 65.0)",False,True,False,False
...,...,...,...,...,...,...,...
9995,British Columbia,Female,"[50.0, 65.0)",False,False,False,True
9996,Ontario,Male,"[65.0, inf)",False,False,False,False
9997,Quebec,Female,"[50.0, 65.0)",False,True,False,False
9998,Ontario,Male,"[65.0, inf)",False,False,True,False
